# Build a model to predict Koc 

The aim of this assignment is to build a model that can predict the soil sorption coefficient logKoc. In contrast to the previous assignment, this time the goal is outperform the state-of-the-art OPERA Koc model:

 - Paper on OPERA models, including the one for Koc: https://doi.org/10.1186/s13321-018-0263-1
 - Performance of the OPERA Koc model on the test set: R2 = 0.71, RMSE = 0.61
 - Model: weighted k-Nearest Neighbors regressor trained on 12 selected PaDEL descriptors (similar to RDKit descriptors)

As you are now a modelling expert, it is open to you which model architecture you use. BUT we ask you to justify your choices! 


#### Tasks:

1) Load the training data and split it into train and test according to the 'Tr_1_Tst_0' column

2) Think about a suitable architecture:
    - Suitable descriptors/fingerprints
    - Model choice (fine-tune a pretrained neural network? GNN? Ensemble method? Gaussian Process? ...?)
    - Consider providing an AD for your model (using an AD metric, or by defining the content of the training data (e.g., organic chemicals with a MW between x and y)
    - Consider providing prediction uncertainty: If you decide to do so, provide uncertainty calibration. If not, explain why.
    
        
3) Train the model on the same training data used in the paper (Tr_1_Tst_0 == 1). 

4) Consider tuning hyperparameters (e.g., using GridSearchCV and on a short list of parameters) 

5) Evaluate model on the test set (Tr_1_Tst_0 == 0), ONLY ONCE ! 

6) Compare your model performance on the test set to the OPERA model. 

#### Questions:
1) Which architecture did you choose, and why?
2) How well does your model perform on the test set? Could you outperform OPERA?
3) Did you add prediction uncertainty, and why? Are the uncertainties well calibrated?
4) Do you provide an AD, and why? How did you define your AD


In [8]:
# import
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer, mean_pinball_loss

from rdkit import Chem
from rdkit.Chem.Descriptors import CalcMolDescriptors

#### 1. Load training data

In [4]:
# Loading the data from the .sdf file
supplier = Chem.SDMolSupplier("../KOC_QR.sdf")
df = pd.DataFrame([
    {
        "SMILES": Chem.MolToSmiles(m),
        **{p: m.GetProp(p) for p in ['preferred_name', 'LogKOC', 'Tr_1_Tst_0']}
    }
    for m in supplier if m is not None
])

In [5]:
# The data is pre-split in training and testing:
df.groupby('Tr_1_Tst_0').count()

,SMILES,preferred_name,LogKOC
Tr_1_Tst_0,,,
0,184,184,184
1,544,544,544


#### 2. Build a supervised model of your choice

In [11]:
# convert the target variable to numeric
df['LogKOC'] = pd.to_numeric(df['LogKOC'])
# Split data in pre-defined training and test
df_train = df[df['Tr_1_Tst_0'] == '1'].copy()
df_test = df[df['Tr_1_Tst_0'] == '0'].copy()

#### Question 1: Which architecture did you choose, and why?
Key considerations:
- The paper states that OPERA's logKoc model may be overfitting, due to a drop in performance when going from the training (R2=0.81) to the test set (R2=0.71). As a reason, they suspect "the biological complexity of the endpoint". We therefore assume that the aleatoric uncertainty in the dataset is high.
- One of the objectives in the development of the OPERA model was to keep the models as simple as possible and to provide maximal interpretability. For this reason, the Koc model was trained on only 12 carefully selected descriptors. Our objective is to outperform OPERA, which is why we can consider a larger and more complex descriptor space, as well as a less transparent model.

In [16]:
# Descriptors chosen: RDKit descriptors, as they cover a wide range of structural features and chemical properties
def RDKit_desc_from_smiles(smiles_list):
    descs = []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        desc = CalcMolDescriptors(mol)
        descs.append(desc)
    return pd.DataFrame(descs)

# load the training data
y_train = df_train['LogKOC'].values # Target variable train
X_train = RDKit_desc_from_smiles(df_train['SMILES'].values) # Calculate molecular descriptors
X_train.dropna(axis="columns", how="any", inplace=True)
print("Number of features for modelling:", len(X_train.columns))

Number of features for modelling: 197


In [30]:
# Model: Quantile Gradient Boosting with optimized hyperparameters

# Define quantiles (e.g., 10% lower quantile = 0.1, median regression = 0.5, 90% upper quantile = 0.9)
quantiles = [0.1, 0.5, 0.9]
models = {}
# We build a model for each quantile, and then optimize them one by one
for q in quantiles:
    gbr = GradientBoostingRegressor(loss='quantile', alpha=q)
    gbr.fit(X_train, y_train)
    models[q] = gbr

# Parameter grid to tune
param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1],
    'max_depth': [2, 3, 4],
}
# Cross-validation strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Optimiz the models
best_models = {}
for q in quantiles:
    # Use pinball loss (appropriate for quantile regression)
    scorer = make_scorer(mean_pinball_loss, alpha=q, greater_is_better=False)
    
    # Grid search
    grid_search = GridSearchCV(
        estimator=models[q],
        param_grid=param_grid,
        scoring=scorer, # Here, we use a custom scoring function
        cv=cv,
        n_jobs=-1, # use all CPU for grid search
    )
    # Fit
    grid_search.fit(X_train, y_train)
    # Results
    print('-'*5,"Quantile:", q, '-'*5)
    print("Best parameters:", grid_search.best_params_)
    print("Best performance (R2):", grid_search.best_score_)
    print("Average performance (R2) for each combination of hyperparameters:\n", grid_search.cv_results_['mean_test_score'])

    # Best model
    best_models[q] = grid_search.best_estimator_

----- Quantile: 0.1 -----
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Best performance (R2): -0.12086926487784792
Average performance (R2) for each combination of hyperparameters:
 [-0.15349653 -0.13889729 -0.13230955 -0.1516058  -0.13553852 -0.1311552
 -0.15086145 -0.13589721 -0.1298757  -0.1217962  -0.12093324 -0.1219941
 -0.1245097  -0.12086926 -0.12334011 -0.1245639  -0.12186655 -0.12464827]
----- Quantile: 0.5 -----
Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 500}
Best performance (R2): -0.21616069152778192
Average performance (R2) for each combination of hyperparameters:
 [-0.327963   -0.25574528 -0.24013309 -0.3180135  -0.24441847 -0.23256883
 -0.3131262  -0.24207924 -0.23038279 -0.22671043 -0.22135354 -0.21616069
 -0.22523335 -0.21909524 -0.21881759 -0.22867812 -0.22279025 -0.22383981]
----- Quantile: 0.9 -----
Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}
Best performance (R2): -0.11493

#### 3. Model performance on test set
--> **TEST ONLY ONCE !**

In [27]:
# load the test data
y_test = df_test['LogKOC'].values # Target variable train
X_test_all = RDKit_desc_from_smiles(df_test['SMILES'].values) # Calculate molecular descriptors
X_test = X_test_all[X_train.columns]

In [31]:
y_pred = {}
for q in quantiles:
    y_pred[q] = best_models[q].predict(X_test)
    
# Compare predicted median with true values
print('Test R²:', r2_score(y_test, y_pred[0.5]))
print('Test RMSE:', root_mean_squared_error(y_test, y_pred[0.5]))

# Are the quantiles well calibrated? If yes, 80% of the true values should lie within the 0.1-0.9 quantile range
above_lower = (y_test >= y_pred[0.1]) # true logP is above the lower prediction threshold
below_upper = (y_test <= y_pred[0.9]) # true logP is below the upper prediction threshold
# --> WITHIN the confidence interval
emp_cov_qrf = np.mean(above_lower & below_upper)
print('Empirical coverage of Quantile RF (10–90%)', emp_cov_qrf)


Test R²: 0.7871245631092813
Test RMSE: 0.5273090122778442
Empirical coverage of Quantile RF (10–90%) 0.7119565217391305


#### Question 2: How well does your model perform on the test set? Could you outperform OPERA?
The performance of our model is better than OPERA on the test set. However, it is also less transparent, as we use Gradient Boosting and >10 times more molecular descriptors than OPERA.

#### Question 3: Did you add prediction uncertainty, and why? Are the uncertainties well calibrated?
I suspected aleatoric uncertainty (experimental noise) and epistemic uncertainty (comparatively small training data set) to be important, and therefore decided to do Quantile Gradient Boosting regression with 10% and 90% quantiles. The empirical coverage is lower than the confidence interval, suggesting that the intervals are slightly overconfident.

In [35]:
# I will provide a range of molecular weight as an applicability domain 
molwt_min = X_train['MolWt'].min()
molwt_max = X_train['MolWt'].max()
print(f'Training data MolWt ranges between {molwt_min} and {molwt_max}')

logkoc_min = y_train.min()
logkoc_max = y_train.max()
print(f'Training data logKoc ranges between {logkoc_min} and {logkoc_max}')

Training data MolWt ranges between 32.042 and 665.014
Training data logKoc ranges between 0.0 and 6.5



#### Question 4: Do you provide an AD, and why? How did you define your AD
As my model provides prediction uncertainty, I decided against providing an AD metric for predictions. I advise users to only apply the model to organic chemicals with a molecular weight between 30 and 700 Da, and to consider the predicted confidence intervals when working with the predicted values.